
# Taller — Titanic (Regresión Logística) con Pipeline + ColumnTransformer (ESQUELETO)

En esta tarea vas a **completar** un flujo profesional de clasificación para predecir `Survived`.

Tendrás tres archivos (en `data/`):
- `titanic_train_70.csv` (con etiquetas)
- `titanic_public_test_with_labels_15.csv` (con etiquetas, para experimentar)
- `titanic_private_test_without_labels_15.csv` (sin etiquetas, para la entrega)

🎯 Objetivo:
- Entrenar y evaluar tu modelo usando **train** y **public test**
- Generar `submission.csv` para el **private test**

✅ Reglas:
- Todo el preprocesamiento debe ir en un **Pipeline**
- Debes usar **Regresión Logística** como modelo final
- No inventes variables mirando el target en el private test (no existe)


## 1. Importar librerías

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/AI_IS/5_linlog_reg"

/content/drive/MyDrive/Courses/AI_IS/5_linlog_reg


In [3]:

import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

import matplotlib.pyplot as plt


## 2. Cargar datasets

In [4]:

DATA_DIR = "data"

train_path = os.path.join(DATA_DIR, "titanic_train_70.csv")
public_path = os.path.join(DATA_DIR, "titanic_public_test_with_labels_15.csv")
private_path = os.path.join(DATA_DIR, "titanic_private_test_without_labels_15.csv")

for p in [train_path, public_path, private_path]:
    assert os.path.exists(p), f"No encuentro el archivo: {p}"

train_df = pd.read_csv(train_path)
public_df = pd.read_csv(public_path)
private_df = pd.read_csv(private_path)

train_df.shape, public_df.shape, private_df.shape


((623, 12), (134, 12), (134, 11))

In [5]:

train_df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,749,0,1,"Marvin, Mr. Daniel Warner",male,19.0,1,0,113773,53.1000,D30,S
1,46,0,3,"Rogers, Mr. William John",male,NaN,0,0,S.C./A.4. 23567,8.0500,NaN,S
2,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
3,634,0,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.0000,NaN,S
4,404,0,3,"Hakkarainen, Mr. Pekka Pietari",male,28.0,1,0,STON/O2. 3101279,15.8500,NaN,S



## 3. Entender las variables (¡antes de modelar!)

Titanic trae columnas típicas como:

- **PassengerId**: identificador (NO explica supervivencia por sí mismo). Suele usarse solo para el submission.
- **Name**: texto con nombre (puede contener títulos como *Mr, Mrs, Miss*).
- **Sex**: sexo (categórica).
- **Age**: edad (numérica, con faltantes).
- **SibSp / Parch**: familiares a bordo (numéricas).
- **Ticket**: identificador de ticket (alta cardinalidad, puede introducir ruido).
- **Fare**: tarifa (numérica).
- **Cabin**: cabina (muchos faltantes, texto).
- **Embarked**: puerto de embarque (categórica).

👉 Tu trabajo: **decidir qué variables ayudan** y cuáles estorban.

### Pistas (no son reglas absolutas)
- Identificadores puros suelen aportar poco: `PassengerId`, `Ticket`.
- Texto libre puede ser difícil: `Name` (a menos que extraigas una señal simple como el título).
- Muchísimos faltantes: `Cabin` (a veces se descarta o se transforma a “tiene cabina: sí/no”).


### 3.1 Revisión rápida de tipos y faltantes

In [6]:

# Tipos de datos
train_df.dtypes


,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [7]:

# Porcentaje de valores faltantes por columna (train)
missing_pct = (train_df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing_pct


,0
Cabin,78.17
Age,19.10
Embarked,0.32
PassengerId,0.00
Name,0.00
Pclass,0.00
Survived,0.00
Sex,0.00
Parch,0.00
SibSp,0.00



### 3.2 Distribución del target

En `train` y `public` existe `Survived`.  
Revisa si hay desbalance (no siempre es grave, pero importa para métricas).


In [8]:

TARGET = "Survived"
train_df[TARGET].value_counts(normalize=True).rename("train")


,train
Survived,
0,0.616372
1,0.383628


In [ ]:

public_df[TARGET].value_counts(normalize=True).rename("public")



## 4. Selección de variables (TODO)

Aquí vas a crear una lista de columnas para **eliminar** o **conservar**.

✅ Lo mínimo recomendado:
- Mantener variables claras: `Sex`, `Age`, `Pclass`, `Fare`, `SibSp`, `Parch`, `Embarked`
- No usar `PassengerId` como predictor (pero lo necesitas para el submission)
- Decidir qué hacer con `Name`, `Ticket`, `Cabin`

### TODO
1) Define una lista `DROP_COLS` con columnas que NO usarás como features.  
2) Crea `X_train`, `y_train`, `X_public`, `y_public`, `X_private` aplicando ese filtro.

💡 Consejo: empieza simple (descarta `Name`, `Ticket`, `Cabin`) y luego experimenta.


In [ ]:

# Columnas que NO usaremos como features
DROP_COLS = ["PassengerId", "Name", "Ticket", "Cabin"]

# Separar features (X) y target (y)
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET]).drop(columns=DROP_COLS)

y_public = public_df[TARGET]
X_public = public_df.drop(columns=[TARGET]).drop(columns=DROP_COLS)

# private_df NO tiene columna Survived
X_private = private_df.drop(columns=[c for c in DROP_COLS if c in private_df.columns])

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_public:", X_public.shape, "y_public:", y_public.shape)
print("X_private:", X_private.shape)
print("\nColumnas usadas:", list(X_train.columns))




## 5. Preprocesamiento con ColumnTransformer (TODO)

Usa:
- Numéricas: imputación + (opcional) escalado
- Categóricas: imputación + OneHotEncoder

### TODO
1) Identifica `num_cols` y `cat_cols` a partir de `X_train`  
2) Construye:
- `numeric_preprocess`
- `categorical_preprocess`
- `preprocess = ColumnTransformer(...)`


In [ ]:
# Identificar columnas numéricas y categóricas
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

# Sub-pipeline numérico: imputar faltantes con la mediana + escalar
numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Sub-pipeline categórico: imputar faltantes con la moda + one-hot
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combinar ambos sub-pipelines
preprocess = ColumnTransformer(transformers=[
    ("num", numeric_preprocess, num_cols),
    ("cat", categorical_preprocess, cat_cols)
])

preprocess


## 6. Pipeline + Regresión Logística (TODO)

Crea un `Pipeline` con:
1) `("preprocess", preprocess)`
2) `("model", LogisticRegression(...))`

### TODO
- Crea `pipe`
- Entrena con `pipe.fit(X_train, y_train)`


In [ ]:

# Crear el modelo de Regresión Logística
model = LogisticRegression(max_iter=1000, random_state=42)

# Pipeline completo: preprocesamiento + modelo
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# Entrenar
pipe.fit(X_train, y_train)

print("Accuracy en train:", round(pipe.score(X_train, y_train), 4))



## 7. Evaluación en el Public Test (TODO)

Calcula:
- `classification_report`
- matriz de confusión
- (opcional) ROC-AUC si usas `predict_proba`

### TODO
- Genera `y_pred_public`
- Muestra métricas


In [ ]:

# Predecir en el public test
y_pred_public = pipe.predict(X_public)

# Classification report
print(classification_report(y_public, y_pred_public, digits=4))

# Matriz de confusión
cm = confusion_matrix(y_public, y_pred_public)
ConfusionMatrixDisplay(cm).plot()
plt.title("Matriz de Confusión — Public Test")
plt.show()

# ROC-AUC
if hasattr(pipe, "predict_proba"):
    y_proba_public = pipe.predict_proba(X_public)[:, 1]
    print("ROC-AUC:", round(roc_auc_score(y_public, y_proba_public), 4))



## 8. Entrenamiento final y submission (TODO)

Cuando decidas tu “mejor versión” del pipeline:

1) (Opcional) Entrena con **train + public** para tener más datos etiquetados.  
2) Predice el **private test sin etiquetas**.  
3) Genera `data/submission.csv` con:
- `PassengerId`
- `Survived` (predicción)

### TODO
- Entrenar final
- Predecir
- Guardar submission


In [ ]:
# Combinar train + public para entrenamiento final
full_train_df = pd.concat([train_df, public_df], axis=0, ignore_index=True)
X_full = full_train_df.drop(columns=[TARGET]).drop(columns=DROP_COLS)
y_full = full_train_df[TARGET]

# Re-entrenar pipeline con todos los datos etiquetados
pipe.fit(X_full, y_full)
print("Accuracy en full train:", round(pipe.score(X_full, y_full), 4))

# Predecir el private test
private_pred = pipe.predict(X_private)

# Generar submission.csv
ID_COL = "PassengerId"
assert ID_COL in private_df.columns, "PassengerId debe existir en el private test"

submission = pd.DataFrame({
    ID_COL: private_df[ID_COL].values,
    TARGET: private_pred
})

out_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Submission guardado en: {out_path}")
print(f"Forma: {submission.shape}")
submission.head()



## 9. Ejercicios (elige al menos 3) — explica con evidencia

Usa métricas del **public test** para justificar tus respuestas (no solo opinión).

1) **Variables**: prueba incluir/excluir `Cabin`, `Name`, `Ticket`. ¿Qué cambia? ¿Por qué crees?
2) **Cabin simple**: crea una variable `HasCabin = Cabin.notna().astype(int)` y elimina `Cabin` original. ¿Mejora?
3) **Título del nombre**: extrae el título (Mr/Mrs/Miss/… ) desde `Name` y úsalo como categórica. ¿Mejora?
4) **Regularización**: prueba `C = 0.1, 1, 10`. ¿Qué pasa con precision/recall?
5) **Balanceo**: prueba `class_weight="balanced"`. ¿Mejora recall? ¿Pierde precisión?
6) **Escalado**: elimina `StandardScaler`. ¿Qué ocurre en regresión logística?

Plantilla para cada ejercicio:
- Cambio:
- Métricas (public):
- Explicación (2–5 líneas):


In [ ]:
# ------- Ejercicio 1: Variables (incluir/excluir Cabin, Name, Ticket) -------
# Cambio: Incluir Ticket y Cabin junto con las demás features
# (necesitas OneHotEncoder con handle_unknown="ignore" para la alta cardinalidad)

DROP_COLS_EX1 = ["PassengerId", "Name"]  # solo quitamos PassengerId y Name
X_train_ex1 = train_df.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX1)
X_public_ex1 = public_df.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX1)

num_cols_ex1 = X_train_ex1.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_ex1 = X_train_ex1.select_dtypes(include=["object"]).columns.tolist()

preprocess_ex1 = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols_ex1),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols_ex1)
])
pipe_ex1 = Pipeline([("preprocess", preprocess_ex1), ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipe_ex1.fit(X_train_ex1, y_train)
y_pred_ex1 = pipe_ex1.predict(X_public_ex1)
print("=== Ejercicio 1: Incluir Ticket y Cabin ===")
print(classification_report(y_public, y_pred_ex1, digits=4))
# Explicación: Ticket y Cabin tienen alta cardinalidad y muchos valores faltantes
# (Cabin ~78% NaN). Incluirlos generalmente NO mejora porque el one-hot crea
# cientos de columnas con poca señal, generando sobreajuste o ruido.


# ------- Ejercicio 2: Cabin → HasCabin -------
train_df_ex2 = train_df.copy()
public_df_ex2 = public_df.copy()
train_df_ex2["HasCabin"] = train_df_ex2["Cabin"].notna().astype(int)
public_df_ex2["HasCabin"] = public_df_ex2["Cabin"].notna().astype(int)

DROP_COLS_EX2 = ["PassengerId", "Name", "Ticket", "Cabin"]
X_train_ex2 = train_df_ex2.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX2)
X_public_ex2 = public_df_ex2.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX2)

num_cols_ex2 = X_train_ex2.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_ex2 = X_train_ex2.select_dtypes(include=["object"]).columns.tolist()

preprocess_ex2 = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols_ex2),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols_ex2)
])
pipe_ex2 = Pipeline([("preprocess", preprocess_ex2), ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipe_ex2.fit(X_train_ex2, y_train)
y_pred_ex2 = pipe_ex2.predict(X_public_ex2)
print("=== Ejercicio 2: HasCabin ===")
print(classification_report(y_public, y_pred_ex2, digits=4))
# Explicación: HasCabin binario captura si el pasajero tenía cabina asignada,
# lo cual correlaciona con la clase socioeconómica. Es más útil que el texto
# original de Cabin porque reduce el ruido de alta cardinalidad a una sola
# señal informativa.


# ------- Ejercicio 3: Título del nombre -------
import re

def extract_title(name):
    match = re.search(r',\s*(\w+)\.', name)
    return match.group(1) if match else "Other"

train_df_ex3 = train_df.copy()
public_df_ex3 = public_df.copy()
train_df_ex3["Title"] = train_df_ex3["Name"].apply(extract_title)
public_df_ex3["Title"] = public_df_ex3["Name"].apply(extract_title)

DROP_COLS_EX3 = ["PassengerId", "Name", "Ticket", "Cabin"]
X_train_ex3 = train_df_ex3.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX3)
X_public_ex3 = public_df_ex3.drop(columns=[TARGET]).drop(columns=DROP_COLS_EX3)

num_cols_ex3 = X_train_ex3.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_ex3 = X_train_ex3.select_dtypes(include=["object"]).columns.tolist()

preprocess_ex3 = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols_ex3),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols_ex3)
])
pipe_ex3 = Pipeline([("preprocess", preprocess_ex3), ("model", LogisticRegression(max_iter=1000, random_state=42))])
pipe_ex3.fit(X_train_ex3, y_train)
y_pred_ex3 = pipe_ex3.predict(X_public_ex3)
print("=== Ejercicio 3: Título del nombre ===")
print(classification_report(y_public, y_pred_ex3, digits=4))
# Explicación: El título (Mr, Mrs, Miss, Master, etc.) captura información
# sobre género, edad aproximada y estatus social que complementa a Sex y Age.
# "Master" indica niños varones, "Mrs" mujeres casadas, lo que puede mejorar
# la predicción al dar más granularidad que solo "male"/"female".


# ------- Ejercicio 4: Regularización (C = 0.1, 1, 10) -------
print("=== Ejercicio 4: Regularización ===")
for c_val in [0.1, 1, 10]:
    pipe_ex4 = Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(C=c_val, max_iter=1000, random_state=42))
    ])
    pipe_ex4.fit(X_train, y_train)
    y_pred_ex4 = pipe_ex4.predict(X_public)
    print(f"\n--- C = {c_val} ---")
    print(classification_report(y_public, y_pred_ex4, digits=4))
# Explicación: C controla la fuerza de regularización (C bajo → más regularización).
# Con C=0.1, el modelo penaliza más los coeficientes grandes, lo que puede
# reducir overfitting pero podría perder recall. Con C=10, el modelo es más
# libre de ajustarse a los datos, pudiendo ganar recall pero arriesgando overfitting.


# ------- Ejercicio 5: class_weight="balanced" -------
pipe_ex5 = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"))
])
pipe_ex5.fit(X_train, y_train)
y_pred_ex5 = pipe_ex5.predict(X_public)
print("=== Ejercicio 5: class_weight='balanced' ===")
print(classification_report(y_public, y_pred_ex5, digits=4))
# Explicación: class_weight="balanced" asigna pesos inversamente proporcionales
# a la frecuencia de cada clase. Como hay ~62% clase 0 y ~38% clase 1,
# "balanced" aumenta la importancia de la clase minoritaria (Survived=1),
# lo que generalmente mejora el recall de clase 1 pero puede reducir su
# precision si empieza a clasificar más falsos positivos.


# ------- Ejercicio 6: Sin StandardScaler -------
numeric_preprocess_no_scaler = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
preprocess_no_scaler = ColumnTransformer(transformers=[
    ("num", numeric_preprocess_no_scaler, num_cols),
    ("cat", categorical_preprocess, cat_cols)
])
pipe_ex6 = Pipeline([
    ("preprocess", preprocess_no_scaler),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
pipe_ex6.fit(X_train, y_train)
y_pred_ex6 = pipe_ex6.predict(X_public)
print("=== Ejercicio 6: Sin StandardScaler ===")
print(classification_report(y_public, y_pred_ex6, digits=4))
# Explicación: La regresión logística con regularización (por defecto penalty='l2')
# es sensible a la escala de las features. Sin escalar, variables con rangos
# grandes (como Fare: 0-512) dominan la regularización, mientras que variables
# con rangos pequeños (como SibSp: 0-8) se regularizanen exceso. El escalado
# iguala las escalas para que la regularización actúe equitativamente.